In [1]:
#imports
import spacy
import scispacy
import timeit
import pandas as pd
import os
from pathlib import Path
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter


/home/exouser/Desktop/bridge-rna/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/exouser/Desktop/bridge-rna/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
is_gpu_available = spacy.prefer_gpu()
print(f"GPU active: {is_gpu_available}")


GPU active: False


In [3]:
vectorizer = CountVectorizer(lowercase=True,)

In [4]:
#load term parser
nlp = spacy.load("en_core_sci_lg")

In [5]:
#change this to live file ingest later

file_path = Path.cwd().parent / "archs4metadata/OSD-100|Mmus_C57-6J_EYE_FLT_Rep1_M23_archs4_top10hits_metadata.csv"

md_df = pd.read_csv(file_path)

idx = defaultdict(set) #may want to cache later if it gets big

In [6]:
def _add_spacy_terms(): # adds spacy terms to dataframe, returns list of flat terms
    md_df['spacey_terms'] = md_df.apply(lambda _: [], axis=1)

    for row in md_df.itertuples():
        doc = nlp(row.geo_summary)
        for ent in doc.ents:
            term = ent.text.split()
            row.spacey_terms.extend(term)
            for t in term:
                idx[t].add(row.gse)
            
    
    return [item for lst in md_df["spacey_terms"] for item in lst]

In [7]:
def _update_idx():
    
    analyzer = vectorizer.build_analyzer()
    new_idx = defaultdict(set)

    for term,doc_ids in idx.items():
        tokenized = analyzer(term)
        for token in tokenized:
            new_idx[token].update(doc_ids)

    return new_idx  


In [8]:
def _add_journals(cl_df):
     cl_df['journals'] = cl_df.apply(lambda _: [], axis=1)
     cl_df["Representation"] = cl_df["Representation"].apply(lambda lst: [x for x in lst if x])

     for row in cl_df.itertuples():
        
        
        for term in row.Representation:
            
            if term in idx:
               row.journals.extend(list(idx[term]))
            else:
                  print(f"term {term} not found in index")
     
     return cl_df      
         

In [10]:

def cluster_sample(): 
    global idx
    flat_terms = _add_spacy_terms()
    vectorizer.fit(flat_terms)
    idx = _update_idx()
    model = SentenceTransformer('allenai/biomed_roberta_base')
    topic_model = BERTopic(vectorizer_model=vectorizer, embedding_model=model)
    topics,probs = topic_model.fit_transform(flat_terms)
    clusters = _add_journals(topic_model.get_topic_info())   
    return topic_model,clusters
    


In [ ]:
def get_cluster_distribution(clusters,cluster_idx):
    journals = clusters.iloc[cluster_idx].journals
    cnts = Counter(journals)
   
    total = len(journals)

    percent_dist =  {name: (count / total) * 100 for name, count in cnts.items()}
    return percent_dist

In [ ]:
def get_cluster_counts(clusters,cluster_idx):
    journals = clusters.iloc[cluster_idx].journals
    cnts = Counter(journals)
    cnts = dict(cnts)
    return cnts

In [ ]:
def get_top_cluster_distribution(clusters,n=5):
    top_clusters = clusters.head(n)
    distributions = {}
    for idx, row in top_clusters.iterrows():
        distributions[row.Topic] = get_cluster_distribution(clusters,idx)
    return distributions

In [ ]:
def get_top_journals(clusters,n=5):
    
    top_clusters = clusters.head(n).iloc[1:] #skip outliers
    overall_cnts = defaultdict(int)
    for idx, row in top_clusters.iterrows():
       # print(f"Processing cluster {row.Topic} at index {idx}")
        journal_counts = get_cluster_counts(clusters,idx)
       # print(f"Journal counts for cluster {row.Topic}: {journal_counts}")
        for journal, count in journal_counts.items():
            #print(f"Adding {count} to overall count for journal {journal}")
            overall_cnts[journal] += count
    
    return overall_cnts

In [20]:
def get_top_journal_distribution(clusters):
    top_cnt_dict = get_top_journals(clusters)
    total = sum(top_cnt_dict.values())
    percent_dist = {name: (count / total) * 100 for name, count in top_cnt_dict.items()}
    return percent_dist


In [21]:
model,clusters = cluster_sample() #47s no gpu

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24785.20it/s]
[transformers] RobertaModel LOAD REPORT from: allenai/biomed_roberta_base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

In [19]:
top_j = get_top_journals(clusters)
jour_dist = get_top_journal_distribution(clusters)

TypeError: cannot do positional indexing on RangeIndex with these indexers [    Topic  Count                                       Name  \
0      -1     48      -1_phagocytosis_tumor_anti_sequencing   
1       0     84                 0_loss_eye_kinase_function   
2       1     39        1_retinal_retina_molecular_receptor   
3       2     38               2_tyro3_efemp1_tug1_efemp1ki   
4       3     31       3_mertk_macrophages_macrophage_doyne   
5       4     28         4_segments_processes_genes_eyecups   
6       5     28           5_mouse_expression_changes_cells   
7       6     27              6_process_models_analysis_set   
8       7     27          7_expressed_neural_whole_modifier   
9       8     22      8_knockout_disease_degeneration_onset   
10      9     20                9_uxt_r345w_leventinese_kos   
11     10     15                    10_gene_genome_genetic_   
12     11     15                          11_traits_trait__   
13     12     15                          

In [ ]:
get_top_cluster_distribution(5) #note: includes outliers as -1 index

{-1: {'GSE205070': 63.63636363636363,
  'GSE210492': 27.27272727272727,
  'GSE124745': 9.090909090909092},
 0: {'GSE205070': 58.333333333333336,
  'GSE210492': 25.0,
  'GSE88819': 8.333333333333332,
  'GSE124745': 8.333333333333332},
 1: {'GSE210492': 27.27272727272727,
  'GSE205070': 54.54545454545454,
  'GSE143281': 18.181818181818183},
 2: {'GSE205070': 18.181818181818183,
  'GSE210492': 27.27272727272727,
  'GSE88819': 9.090909090909092,
  'GSE124745': 27.27272727272727,
  'GSE143281': 18.181818181818183},
 3: {'GSE205070': 60.0, 'GSE210492': 20.0, 'GSE88819': 20.0}}